In [1]:
import os, glob, random, cv2, numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

✅ Device: cuda


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:
        subindent = "  " * (level + 1)
        for f in files[:3]:
            print(f"{subindent}{f}")

input/
  datasets/
    kmader/
      finding-lungs-in-ct-data/
        2d_masks.zip
        2d_images.zip
        lung_stats.csv
        2d_images/
          ID_0055_Z_0122.tif
          ID_0144_Z_0224.tif
          ID_0111_Z_0078.tif
        2d_masks/
          ID_0055_Z_0122.tif
          ID_0144_Z_0224.tif
          ID_0111_Z_0078.tif
        3d_images/
          IMG_0059.nii.gz
          MASK_0059.nii.gz
          MASK_0002.nii.gz


In [4]:
image_files = sorted(glob.glob("/kaggle/input/datasets/kmader/finding-lungs-in-ct-data/2d_images/*.tif"))
mask_files  = sorted(glob.glob("/kaggle/input/datasets/kmader/finding-lungs-in-ct-data/2d_masks/*.tif"))

print(f"🖼️  Images : {len(image_files)}")
print(f"🎭 Masks  : {len(mask_files)}")

🖼️  Images : 267
🎭 Masks  : 267


In [22]:
def preprocess(img_path, mask_path, size=256):
    img  = np.array(Image.open(img_path))
    mask = np.array(Image.open(mask_path))

    if img.dtype == np.uint16:
        img = (img / 65535.0 * 255).astype(np.uint8)
    else:
        img = img.astype(np.uint8)

    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    if len(mask.shape) == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)

    img  = cv2.resize(img,  (size, size))
    mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img   = clahe.apply(img).astype(np.float32) / 255.0
    mask  = (mask > 127).astype(np.float32)

    return img, mask

In [23]:
class LungDataset(Dataset):
    def __init__(self, img_files, mask_files):
        self.imgs  = img_files
        self.masks = mask_files

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img, mask = preprocess(self.imgs[idx], self.masks[idx])
        img  = torch.tensor(img).unsqueeze(0)   # (1, 256, 256)
        mask = torch.tensor(mask).unsqueeze(0)  # (1, 256, 256)
        return img, mask

# Train/Val split
split     = int(0.85 * len(image_files))
train_ds  = LungDataset(image_files[:split], mask_files[:split])
val_ds    = LungDataset(image_files[split:], mask_files[split:])
train_dl  = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl    = DataLoader(val_ds,   batch_size=16)

print(f"✅ Train: {len(train_ds)} | Val: {len(val_ds)}")

✅ Train: 226 | Val: 41


In [24]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(DoubleConv(1, 64));   self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(DoubleConv(64, 128)); self.pool2 = nn.MaxPool2d(2)
        self.enc3 = nn.Sequential(DoubleConv(128, 256));self.pool3 = nn.MaxPool2d(2)
        self.enc4 = nn.Sequential(DoubleConv(256, 512));self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.up4   = nn.ConvTranspose2d(1024, 512, 2, 2); self.dec4 = DoubleConv(1024, 512)
        self.up3   = nn.ConvTranspose2d(512, 256, 2, 2);  self.dec3 = DoubleConv(512, 256)
        self.up2   = nn.ConvTranspose2d(256, 128, 2, 2);  self.dec2 = DoubleConv(256, 128)
        self.up1   = nn.ConvTranspose2d(128, 64, 2, 2);   self.dec1 = DoubleConv(128, 64)
        self.out   = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        s1 = self.enc1(x); x = self.pool1(s1)
        s2 = self.enc2(x); x = self.pool2(s2)
        s3 = self.enc3(x); x = self.pool3(s3)
        s4 = self.enc4(x); x = self.pool4(s4)
        x  = self.bottleneck(x)
        x  = self.dec4(torch.cat([self.up4(x), s4], 1))
        x  = self.dec3(torch.cat([self.up3(x), s3], 1))
        x  = self.dec2(torch.cat([self.up2(x), s2], 1))
        x  = self.dec1(torch.cat([self.up1(x), s1], 1))
        return torch.sigmoid(self.out(x))

model = UNet().to(device)
print(f"✅ Model ready! Params: {sum(p.numel() for p in model.parameters()):,}")

✅ Model ready! Params: 31,036,481


In [25]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

def dice_loss(pred, target, eps=1e-6):
    inter = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()

def bce_dice(pred, target):
    return nn.BCELoss()(pred, target) + dice_loss(pred, target)

best_val = float('inf')
EPOCHS   = 30

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for imgs, masks in train_dl:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = bce_dice(model(imgs), masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Val
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in val_dl:
            imgs, masks = imgs.to(device), masks.to(device)
            val_loss += bce_dice(model(imgs), masks).item()

    train_loss /= len(train_dl)
    val_loss   /= len(val_dl)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "/kaggle/working/best_unet_model.pth")
        print(f"   💾 Saved! Best val: {best_val:.4f}")

print("✅ Training complete!")

Epoch 01/30 | Train: 1.1677 | Val: 76.0896
   💾 Saved! Best val: 76.0896
Epoch 02/30 | Train: 0.7777 | Val: 0.7857
   💾 Saved! Best val: 0.7857
Epoch 03/30 | Train: 0.6355 | Val: 0.6859
   💾 Saved! Best val: 0.6859
Epoch 04/30 | Train: 0.5498 | Val: 0.7652
Epoch 05/30 | Train: 0.4896 | Val: 1.1728
Epoch 06/30 | Train: 0.4214 | Val: 1.2867
Epoch 07/30 | Train: 0.3949 | Val: 1.0350
Epoch 08/30 | Train: 0.3333 | Val: 1.4626
Epoch 09/30 | Train: 0.2924 | Val: 1.8817
Epoch 10/30 | Train: 0.2714 | Val: 0.6537
   💾 Saved! Best val: 0.6537
Epoch 11/30 | Train: 0.2555 | Val: 1.9008
Epoch 12/30 | Train: 0.2618 | Val: 1.6627
Epoch 13/30 | Train: 0.2483 | Val: 1.6757
Epoch 14/30 | Train: 0.2204 | Val: 0.8680
Epoch 15/30 | Train: 0.2021 | Val: 1.8097
Epoch 16/30 | Train: 0.1980 | Val: 0.7554
Epoch 17/30 | Train: 0.1860 | Val: 0.6677
Epoch 18/30 | Train: 0.1734 | Val: 1.4489
Epoch 19/30 | Train: 0.1635 | Val: 0.4900
   💾 Saved! Best val: 0.4900
Epoch 20/30 | Train: 0.1590 | Val: 0.5941
Epoch 21/30 |

In [26]:
def dice_score(pred, target, threshold=0.5, eps=1e-6):
    pred = (pred > threshold).float()
    inter = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return ((2 * inter + eps) / (union + eps)).mean().item()

model.load_state_dict(torch.load("/kaggle/working/best_unet_model.pth"))
model.eval()

scores = []
with torch.no_grad():
    for imgs, masks in val_dl:
        imgs, masks = imgs.to(device), masks.to(device)
        scores.append(dice_score(model(imgs), masks))

print(f"🎯 Mean Dice Score: {np.mean(scores):.4f}")

🎯 Mean Dice Score: 0.8172
